<a href="https://colab.research.google.com/github/tillyrcohen-code/python-fundamentals/blob/main/kent_heritage_spatial_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kent Archaeological & Heritage Spatial Intelligence Pipeline
**My Bachelor's Degree Focus:** Classical & Archaeological Studies at the University of Kent  
**Project Goal:** To build an automated data pipeline that absorbs, cleans, stores, and maps archaeological heritage records.

In [35]:
# Import core libraries needed
import sqlite3  # database storage tool
import numpy as np  # random number generator
import pandas as pd  # digital spreadsheet manager

print("Tools successfully imported. Ready to generate data.")

Tools successfully imported. Ready to generate data.


In [36]:
# Random seed so random data looks consistent every time
np.random.seed(42)

# Manufacture 100 fake artifact records mimicking regional heritage scheme
raw_data = {
    "artifact_id": [f"KENT-PAS-{i:04d}" for i in range(1, 101)],
    "broadperiod": np.random.choice(
        ["ROMAN", "Roman Period", "IRON AGE", "Iron age", "MEDIEVAL", None],
        100,
        p=[0.35, 0.1, 0.25, 0.05, 0.2, 0.05]
    ),
    "material": np.random.choice(
        ["Copper Alloy", "Bronze", "gold", "Silver", "POTTERY", None],
        100,
        p=[0.3, 0.2, 0.15, 0.15, 0.1, 0.1]
    ),
    "lat": np.random.uniform(51.05, 51.45, 100), # Random Kent coordinates
    "lon": np.random.uniform(0.40, 1.40, 100)
}

# Convert raw dictionary
df_raw = pd.DataFrame(raw_data)

# Add missing values into latitude - simulate real messy data
df_raw.loc[df_raw["artifact_id"].sample(4).index, "lat"] = np.nan

print(f"Generated {len(df_raw)} raw archaeological records with simulated messiness.")
print("\nPreview of the messy data:")
df_raw.head()

Generated 100 raw archaeological records with simulated messiness.

Preview of the messy data:


,artifact_id,broadperiod,material,lat,lon
0,KENT-PAS-0001,Roman Period,Copper Alloy,51.306813,0.451682
1,KENT-PAS-0002,None,gold,51.083656,0.931355
2,KENT-PAS-0003,Iron age,Bronze,51.114651,0.940635
3,KENT-PAS-0004,IRON AGE,gold,51.409422,1.037430
4,KENT-PAS-0005,ROMAN,None,51.292572,1.126091


In [37]:
# Make copy of raw data so it doesn't overwrite original mess
df_clean = df_raw.copy()

# Drop rows missing latitude coordinates
df_clean = df_clean.dropna(subset=["lat"])

# Fix lowercase and mixed spellings
period_mapping = {
    "ROMAN": "ROMAN",
    "Roman Period": "ROMAN",
    "IRON AGE": "IRON AGE",
    "Iron age": "IRON AGE",
    "MEDIEVAL": "MEDIEVAL"
}
df_clean["broadperiod"] = df_clean["broadperiod"].map(period_mapping).fillna("UNKNOWN")

material_mapping = {
    "Copper Alloy": "COPPER ALLOY",
    "Bronze": "COPPER ALLOY",  # Grouped as they represent same broader material category of copper-based metalwork
    "gold": "GOLD",
    "Silver": "SILVER",
    "POTTERY": "CERAMIC"
}
df_clean["material"] = df_clean["material"].map(material_mapping).fillna("OTHER")

print(f"Cleaning complete. Cleaned dataset now contains {len(df_clean)} records.")
print("\nPreview of the newly cleaned data:")
df_clean.head()

Cleaning complete. Cleaned dataset now contains 96 records.

Preview of the newly cleaned data:


,artifact_id,broadperiod,material,lat,lon
0,KENT-PAS-0001,ROMAN,COPPER ALLOY,51.306813,0.451682
1,KENT-PAS-0002,UNKNOWN,GOLD,51.083656,0.931355
2,KENT-PAS-0003,IRON AGE,COPPER ALLOY,51.114651,0.940635
3,KENT-PAS-0004,IRON AGE,GOLD,51.409422,1.037430
4,KENT-PAS-0005,ROMAN,OTHER,51.292572,1.126091


In [38]:
import folium
from folium.plugins import MarkerCluster

# Check column names lowercase
df_clean.columns = [col.lower() for col in df_clean.columns]

# Centre map over Kent using public Esri tile layer - no watermarks
kent_centre = [51.2787, 0.5217]
m = folium.Map(
    location=kent_centre,
    zoom_start=9,
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles &copy; Esri"
)

# Add marker cluster plugin
marker_cluster = MarkerCluster().add_to(m)

# Colour code different historical periods
color_map = {
    "ROMAN": "red",
    "IRON AGE": "orange",
    "MEDIEVAL": "blue",
    "UNKNOWN": "grey"
}

# Loop through each clean record and add pin to map
for idx, row in df_clean.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=6,
        popup=f"<b>ID:</b> {row['artifact_id']}<br><b>Period:</b> {row['broadperiod']}<br><b>Material:</b> {row['material']}",
        color=color_map.get(row["broadperiod"], "black"),
        fill=True,
        fill_opacity=0.8
    ).add_to(marker_cluster)

print("Interactive geospatial map successfully generated")
# Display clean map
m


Interactive geospatial map successfully generated


In [39]:
import folium
from folium.plugins import MarkerCluster

df_clean.columns = [col.lower() for col in df_clean.columns]


kent_centre = [51.2787, 0.5217]
m = folium.Map(
    location=kent_centre,
    zoom_start=9,
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles &copy; Esri"
)


marker_cluster = MarkerCluster().add_to(m)


color_map = {
    "ROMAN": "red",
    "IRON AGE": "orange",
    "MEDIEVAL": "blue",
    "UNKNOWN": "grey"
}


for idx, row in df_clean.iterrows():
    # HTML content for click popup
    popup_html = f"""
    <div style="font-family: sans-serif; font-size: 13px; line-height: 1.4;">
        <b style="colour: #333;">Artifact ID:</b> {row['artifact_id']}<br>
        <b>Period:</b> {row['broadperiod']}<br>
        <b>Material:</b> {row['material']}<br>
        <b>Coordinates:</b> {row['lat']:.4f}, {row['lon']:.4f}
    </div>
    """

    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=7,
        popup=folium.Popup(popup_html, max_width=250),
        tooltip=f"{row['artifact_id']} ({row['broadperiod']}) - Click for details",
        color=color_map.get(row["broadperiod"], "black"),
        fill=True,
        fill_opacity=0.8
    ).add_to(marker_cluster)

print("Interactive map generated successfully")
# Display map
m

# Save map as standalone HTML file
m.save("kent_archaeological_map.html")
print("Map successfully exported to kent_archaeological_map.html.")

# Automatically download file to computer
from google.colab import files
files.download("kent_archaeological_map.html")

Interactive map generated successfully
Map successfully exported to kent_archaeological_map.html.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>